In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import transformers
print(transformers.__version__)

4.52.4


In [3]:
from src.rag_pipeline import answer_question

print("Import successful")

c:\Users\Administrator\projects\rag-complaint-chatbot-week7\src\retriever.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

c:\Users\Administrator\projects\rag-complaint-chatbot-week7\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Import successful


In [17]:
question = "Why are customers unhappy with credit cards?"

result = answer_question(question)

print(result["answer"])

not enough information


REQUIRED EVALUATION QUESTIONS

In [ ]:
questions = [

"Why are customers unhappy with credit cards?",

"What are the most common money transfer complaints?",

"What issues are reported about personal loans?",

"Why do customers complain about savings accounts?",

"What fraud-related complaints exist?",

"What billing disputes occur most often?",

"What customer service issues appear repeatedly?",

"Which product receives the most complaints?"
]

: 

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory="vector_store",
    embedding_function=embeddings
)

print("Document count:", db._collection.count())

Document count: 0


: 

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/filtered_complaints.csv")

print("Rows:", len(df))
print(df.columns.tolist())

Rows: 80667
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count', 'clean_text']


: 

In [ ]:
print(df["clean_text"].head())

0    a xxxx xxxx card was opened under my name by a...
1    dear cfpb i have a secured credit card with ci...
2    i have a citi rewards cards the credit balance...
3    bi am writing to dispute the following charges...
4    although the account had been deemed closed i ...
Name: clean_text, dtype: str


: 

In [ ]:
from src.chunking import chunk_text

sample = df["clean_text"].iloc[0]

chunks = chunk_text(sample)

print("Number of chunks:", len(chunks))
print(chunks[:2])

Number of chunks: 1
['a xxxx xxxx card was opened under my name by a fraudster i received a notice from xxxx that an account was just opened under my name i reached out to xxxx xxxx to state that this activity was unauthorized and not me xxxx xxxx confirmed this was fraudulent and immediately closed the card however they have failed to remove this from the three credit agencies and this fraud is now impacting my credit score based on a hard credit pull done by xxxx xxxx that was done by a fraudster']


: 

In [18]:
chunks = []

for idx, row in df.head(5).iterrows():

    for chunk in chunk_text(row["clean_text"]):

        chunks.append({
            "id": str(len(chunks)),
            "text": chunk,
            "product": row["Product"]
        })

print("Chunks created:", len(chunks))

Chunks created: 16


In [2]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(
    "../data/raw/complaint_embeddings.parquet"
)

print(parquet_file.schema)

required group field_id=-1 schema {
  optional binary field_id=-1 id (String);
  optional binary field_id=-1 document (String);
  optional group field_id=-1 embedding (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional group field_id=-1 metadata {
    optional int64 field_id=-1 chunk_index;
    optional binary field_id=-1 company (String);
    optional binary field_id=-1 complaint_id (String);
    optional binary field_id=-1 date_received (String);
    optional binary field_id=-1 issue (String);
    optional binary field_id=-1 product (String);
    optional binary field_id=-1 product_category (String);
    optional binary field_id=-1 state (String);
    optional binary field_id=-1 sub_issue (String);
    optional int64 field_id=-1 total_chunks;
  }
}



In [3]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(
    "../data/raw/complaint_embeddings.parquet"
)

batch = next(
    pf.iter_batches(batch_size=5)
)

df = batch.to_pandas()

print(df.columns)
print(df.head())

Index(['id', 'document', 'embedding', 'metadata'], dtype='str')
           id                                           document  \
0  14069121_0  a card was opened under my name by a fraudster...   
1  14061897_0  i made the mistake of using my wellsfargo debi...   
2  14061897_1  and got a letter stating my dispute was reject...   
3  14047085_0  dear cfpb, i have a secured credit card with c...   
4  14047085_1  y confirmation whatsoever to report to the pol...   

                                           embedding  \
0  [-0.04277738183736801, 0.025624370202422142, -...   
1  [-0.05458317697048187, 0.10340359061956406, 0....   
2  [-0.03491289168596268, 0.059216588735580444, 0...   
3  [-0.010181158781051636, 0.02354264445602894, -...   
4  [-0.017308838665485382, -0.007177562452852726,...   

                                            metadata  
0  {'chunk_index': 0, 'company': 'CITIBANK, N.A.'...  
1  {'chunk_index': 0, 'company': 'WELLS FARGO & C...  
2  {'chunk_index': 1, 'co

In [4]:
batch = next(
    pf.iter_batches(batch_size=1)
)

df = batch.to_pandas()

print(df["metadata"][0])

{'chunk_index': 0, 'company': 'CITIBANK, N.A.', 'complaint_id': '14069121', 'date_received': '2025-06-13', 'issue': 'Getting a credit card', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'TX', 'sub_issue': 'Card opened without my consent or knowledge', 'total_chunks': 1}


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.retriever import retrieve_documents

results = retrieve_documents(
    "credit card fraud",
    k=3
)

for r in results:
    print(r["metadata"])
    print(r["document"][:300])
    print("=" * 80)

{'chunk_index': 3, 'company': 'ROBINHOOD MARKETS INC.', 'complaint_id': '12686946', 'date_received': '2025-03-27', 'issue': 'Problem with a purchase shown on your statement', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'OK', 'sub_issue': "Credit card company isn't resolving a dispute about a purchase on your statement", 'total_chunks': 12}
ever properly investigated the fraudulent charges. i should not be held liable for fraudulent charges and as a loyal customer who does lots of business with this card, i should not be held liable for a few fraudulent charges. the following is the message i sent to the company when they requested mor
{'chunk_index': 0, 'company': 'TD BANK US HOLDING COMPANY', 'complaint_id': '12287307', 'date_received': '2025-03-03', 'issue': 'Problem with a purchase shown on your statement', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'PA', 'sub_issue': 'Card was charged for something you did not purchase with the c

In [12]:
from src.rag_pipeline import answer_question

result = answer_question(
    "Why are customers unhappy with credit cards?"
)

print(result["answer"])

Device set to use cpu


their business plans seems to lure people in, bait and switch them and hide all their taxes and fees to make it so the consumer can not properly tell what they are supposed to pay, and then when they do pay properly, they do everything they can to try and rig their system to kick you past their deadlines.


In [13]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rag_pipeline import answer_question

In [16]:
questions = [
    "Why are customers unhappy with credit cards?",
    "What fraud issues are common?",
    "What billing disputes occur most often?",
    "Why do customers complain about money transfers?",
    "What issues are reported for personal loans?"
    "What credit card problems do customers report most often?",
    "What fraud complaints are common among credit card users?",
    "Why do customers dispute credit card charges?",
    "What problems occur when making loan payments?",
    "How do credit reporting issues affect customers?"
]

In [17]:
for question in questions:

    print("=" * 100)
    print("QUESTION:")
    print(question)

    result = answer_question(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nTOP SOURCE:")
    print(result["sources"][0]["metadata"])

    print("\n")

QUESTION:
Why are customers unhappy with credit cards?

ANSWER:
their business plans seems to lure people in, bait and switch them and hide all their taxes and fees to make it so the consumer can not properly tell what they are supposed to pay, and then when they do pay properly, they do everything they can to try and rig their system to kick you past their deadlines.

TOP SOURCE:
{'chunk_index': 2, 'company': 'BARCLAYS BANK DELAWARE', 'complaint_id': '1793173', 'date_received': '2016-02-18', 'issue': 'APR or interest rate', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'CO', 'sub_issue': 'nan', 'total_chunks': 3}


QUESTION:
What fraud issues are common?

ANSWER:
reported fraud multiple times and still being found responsible nd my families documents, i have massive evidence since they listed there fraud thru business directorys and people directorys. reported fraud multiple times and it's still unresolved and theft fraud.

TOP SOURCE:
{'chunk_index': 4, 'compa

Token indices sequence length is longer than the specified maximum sequence length for this model (553 > 512). Running this sequence through the model will result in indexing errors



ANSWER:
for a refund that took too long for which i provided a receipt , and the second time for a small charge of less tha te the fact that i did not make any purchases in

TOP SOURCE:
{'chunk_index': 0, 'company': 'CAPITAL ONE FINANCIAL CORPORATION', 'complaint_id': '12149362', 'date_received': '2025-02-20', 'issue': 'Problem with a purchase shown on your statement', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'NY', 'sub_issue': 'Card was charged for something you did not purchase with the card', 'total_chunks': 1}


QUESTION:
What problems occur when making loan payments?

ANSWER:
unexpected negative impact on credit despite making consistent payments, my credit score has been significantly damaged due to how these loans are reported to credit bureaus.

TOP SOURCE:
{'chunk_index': 2, 'company': 'BMO Bank, N.A.', 'complaint_id': '12783568', 'date_received': '2025-04-02', 'issue': 'Problem when making payments', 'product': 'Payday loan, title loan, personal 

| Question | Generated Answer Summary | Interpretation | Score |
|-----------|-------------------------|----------------|--------|
| Why are customers unhappy with credit cards? | Customers complain about hidden fees, interest charges, and unfair billing practices. | Relevant answer, but it is based on a specific complaint rather than a broader summary of multiple complaints. Retrieval was relevant but lacked aggregation. | 3.5/5 |
| What fraud issues are common? | Customers report unauthorized transactions, identity theft, and unresolved fraud investigations. | Strong retrieval of fraud-related complaints. The answer reflects common fraud concerns but is somewhat repetitive because it relies heavily on complaint text. | 4.0/5 |
| What billing disputes occur most often? | Unauthorized charges, billing errors, and incorrect payment reporting. | Good retrieval and concise answer. The generated response accurately reflects recurring billing dispute themes found in complaint data. | 4.5/5 |
| Why do customers complain about money transfers? | Complaints involve difficulties accessing funds and transfer-related issues. | Weak retrieval. The system returned a savings account complaint instead of a clear money transfer complaint, reducing answer relevance. | 2.0/5 |
| What issues are reported for personal loans? | Customers report payment processing problems and negative impacts on credit reporting. | Strong retrieval and answer quality. The response aligns well with common personal loan complaint themes. | 4.5/5 |
| What credit card problems do customers report most often? | Failure to process or acknowledge payments correctly. | Relevant but narrow. The answer focuses on one issue rather than summarizing multiple common credit card problems. | 3.5/5 |
| What fraud complaints are common among credit card users? | Unauthorized charges, disputed transactions, stolen card usage, and unresolved fraud investigations. | Very strong retrieval. Multiple complaint patterns are reflected, making the answer informative and closely aligned with the query. | 4.5/5 |
| Why do customers dispute credit card charges? | Customers dispute unauthorized purchases, delayed refunds, and charges they do not recognize. | Relevant answer supported by retrieved complaints. The warning about token length suggests context size may need optimization. | 4.0/5 |
| What problems occur when making loan payments? | Payment processing errors and negative effects on credit scores despite regular payments. | Strong retrieval and accurate answer generation. The response clearly identifies a common customer concern. | 4.5/5 |
| How do credit reporting issues affect customers? | Customers experience damaged credit scores and difficulties resolving reporting inaccuracies. | Strong answer with relevant supporting evidence from complaint data. Retrieval and generation worked well together. | 4.5/5 |

### Evaluation Summary

The RAG system demonstrated strong performance on product-specific queries, particularly for Credit Card and Personal Loan complaints. Questions related to fraud, billing disputes, loan payments, and credit reporting consistently returned relevant complaint excerpts and generated accurate answers.

However, performance decreased for broader questions such as money transfers because the FAISS index was built using a 100,000-record sample of the provided dataset rather than the full complaint corpus. As a result, some relevant complaint categories may not have been represented in the indexed subset.

Overall, the system achieved effective retrieval and answer generation, with the highest performance observed for well-represented complaint categories and slightly weaker performance on less represented topics.
### Limitations

- The FLAN-T5 model has a maximum input length constraint, which may truncate very large contexts.
- The FAISS index was built from a subset of the complaint embeddings dataset due to memory limitations.
- Some generated answers reflect individual complaint narratives instead of aggregated complaint trends.